<a href="https://colab.research.google.com/github/anudaindu/Mobitel_Alarm_Fault_Prediction/blob/main/alarm_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Importing Libraries

In [8]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

import openpyxl # Added for reading .xlsx files

Importing Data sets

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
file_path = '/content/drive/MyDrive/MobitelData/AlarmLogs20260803113740030_9.xlsx'
try:
    df = pd.read_excel(file_path)
    print(f"Dataset loaded successfully from {file_path}")
    display(df.head())
except FileNotFoundError:
    print(f"Error: The file '{file_path}' was not found. Please check the path and try again.")
except Exception as e:
    print(f"An error occurred while loading the dataset: {e}")

/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Dataset loaded successfully from /content/drive/MyDrive/MobitelData/AlarmLogs20260803113740030_9.xlsx


,,Severity,Occurred On (NT),Cleared On (NT),Alarm ID,MO Name,Comment,Name,Location Information,Additional Information,...,RRU Name,Received On (ST),Request ID,Subnet,Toggling Times,Trouble Ticket ID,Type,User Label,eNodeB ID,gNodeB ID
0,Root alarm,Major,2026-07-07 09:48:01,2026-07-07 11:47:31,25952,BDBAD2_GUL00_BADULLA _HUCH_IP_Ba,-,User Plane Path Fault,"User Plane Host ID=140, PEERIP=69.16.42.6, USE...","RAT_INFO=GL, AFFECTED_RAT=GL, DID=NULL, Cumula...",...,-,2026-07-07 09:49:32,-,ROOT/RAN Subnet/BTS/BTS3900 - Co-MPT/GUL,0,-,Trunk system,-,2704,-
1,-,Major,2026-07-07 09:49:08,2026-07-07 09:49:10,25600,GLPIT2_G_Pitigala2__9__Ab,-,Monitoring Device Maintenance Link Failure,"Cabinet No.=0, Subrack No.=7, Slot No.=0, Boar...","RAT_INFO=G, AFFECTED_RAT=G, DID=AUTODID_201808...",...,-,2026-07-07 09:49:34,-,ROOT/RAN Subnet/MBSC,0,-,Communication system,-,-,-
2,-,Major,2026-07-07 09:49:14,2026-07-07 09:49:16,25600,GLPIT2_G_Pitigala2__9__Ab,-,Monitoring Device Maintenance Link Failure,"Cabinet No.=0, Subrack No.=7, Slot No.=0, Boar...","RAT_INFO=G, AFFECTED_RAT=G, DID=AUTODID_201808...",...,-,2026-07-07 09:49:40,-,ROOT/RAN Subnet/MBSC,0,-,Communication system,-,-,-
3,-,Major,2026-07-07 09:49:19,2026-07-07 09:49:21,25600,GLPIT2_G_Pitigala2__9__Ab,-,Monitoring Device Maintenance Link Failure,"Cabinet No.=0, Subrack No.=7, Slot No.=0, Boar...","RAT_INFO=G, AFFECTED_RAT=G, DID=AUTODID_201808...",...,-,2026-07-07 09:49:44,-,ROOT/RAN Subnet/MBSC,0,-,Communication system,-,-,-
4,-,Major,2026-07-07 09:49:23,2026-07-07 09:49:25,25600,GLPIT2_G_Pitigala2__9__Ab,-,Monitoring Device Maintenance Link Failure,"Cabinet No.=0, Subrack No.=7, Slot No.=0, Boar...","RAT_INFO=G, AFFECTED_RAT=G, DID=AUTODID_201808...",...,-,2026-07-07 09:49:48,-,ROOT/RAN Subnet/MBSC,0,-,Communication system,-,-,-


Basic Analysis of Data

In [16]:
df.head()

,,Severity,Occurred On (NT),Cleared On (NT),Alarm ID,MO Name,Comment,Name,Location Information,Additional Information,...,RRU Name,Received On (ST),Request ID,Subnet,Toggling Times,Trouble Ticket ID,Type,User Label,eNodeB ID,gNodeB ID
0,Root alarm,Major,2026-07-07 09:48:01,2026-07-07 11:47:31,25952,BDBAD2_GUL00_BADULLA _HUCH_IP_Ba,-,User Plane Path Fault,"User Plane Host ID=140, PEERIP=69.16.42.6, USE...","RAT_INFO=GL, AFFECTED_RAT=GL, DID=NULL, Cumula...",...,-,2026-07-07 09:49:32,-,ROOT/RAN Subnet/BTS/BTS3900 - Co-MPT/GUL,0,-,Trunk system,-,2704,-
1,-,Major,2026-07-07 09:49:08,2026-07-07 09:49:10,25600,GLPIT2_G_Pitigala2__9__Ab,-,Monitoring Device Maintenance Link Failure,"Cabinet No.=0, Subrack No.=7, Slot No.=0, Boar...","RAT_INFO=G, AFFECTED_RAT=G, DID=AUTODID_201808...",...,-,2026-07-07 09:49:34,-,ROOT/RAN Subnet/MBSC,0,-,Communication system,-,-,-
2,-,Major,2026-07-07 09:49:14,2026-07-07 09:49:16,25600,GLPIT2_G_Pitigala2__9__Ab,-,Monitoring Device Maintenance Link Failure,"Cabinet No.=0, Subrack No.=7, Slot No.=0, Boar...","RAT_INFO=G, AFFECTED_RAT=G, DID=AUTODID_201808...",...,-,2026-07-07 09:49:40,-,ROOT/RAN Subnet/MBSC,0,-,Communication system,-,-,-
3,-,Major,2026-07-07 09:49:19,2026-07-07 09:49:21,25600,GLPIT2_G_Pitigala2__9__Ab,-,Monitoring Device Maintenance Link Failure,"Cabinet No.=0, Subrack No.=7, Slot No.=0, Boar...","RAT_INFO=G, AFFECTED_RAT=G, DID=AUTODID_201808...",...,-,2026-07-07 09:49:44,-,ROOT/RAN Subnet/MBSC,0,-,Communication system,-,-,-
4,-,Major,2026-07-07 09:49:23,2026-07-07 09:49:25,25600,GLPIT2_G_Pitigala2__9__Ab,-,Monitoring Device Maintenance Link Failure,"Cabinet No.=0, Subrack No.=7, Slot No.=0, Boar...","RAT_INFO=G, AFFECTED_RAT=G, DID=AUTODID_201808...",...,-,2026-07-07 09:49:48,-,ROOT/RAN Subnet/MBSC,0,-,Communication system,-,-,-


In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 44 columns):
 #   Column                         Non-Null Count   Dtype 
---  ------                         --------------   ----- 
 0                                  100000 non-null  object
 1   Severity                       100000 non-null  object
 2   Occurred On (NT)               100000 non-null  object
 3   Cleared On (NT)                100000 non-null  object
 4   Alarm ID                       100000 non-null  int64 
 5   MO Name                        100000 non-null  object
 6   Comment                        100000 non-null  object
 7   Name                           100000 non-null  object
 8   Location Information           100000 non-null  object
 9   Additional Information         100000 non-null  object
 10  Alarm Duration                 100000 non-null  object
 11  Alarm Source                   100000 non-null  object
 12  Acknowledged By                100000 non-nul

In [18]:
df.shape

(100000, 44)

In [19]:
df.columns

Index([' ', 'Severity', 'Occurred On (NT)', 'Cleared On (NT)', 'Alarm ID',
       'MO Name', 'Comment', 'Name', 'Location Information',
       'Additional Information', 'Alarm Duration', 'Alarm Source',
       'Acknowledged By', 'Acknowledged On (ST)', 'Acknowledgement Status',
       'Associated Alarm Group ID', 'Auto Clear', 'BBU Name',
       'Clearance Status', 'Clearance Type', 'Cleared By',
       'Common Alarm Identifier', 'Dev Root CSN',
       'Equipment Alarm Serial Number', 'External Resource ID', 'Link Name',
       'Link Type', 'Log Serial Number', 'Maintenance Region',
       'Maintenance Status', 'NE Address', 'NE Type', 'Object Type',
       'Operation Impact Flag', 'RRU Name', 'Received On (ST)', 'Request ID',
       'Subnet', 'Toggling Times', 'Trouble Ticket ID', 'Type', 'User Label',
       'eNodeB ID', 'gNodeB ID'],
      dtype='object')

Data Cleaning

In [20]:
df.isnull().sum()

,0
,0
Severity,0
Occurred On (NT),0
Cleared On (NT),0
Alarm ID,0
MO Name,0
Comment,0
Name,0
Location Information,0
Additional Information,0


In [21]:
df.duplicated().sum()

np.int64(0)

In [22]:
df = df.drop_duplicates()

To treat `'-'` as missing values, we'll replace all occurrences of `'-'` in the DataFrame with `np.nan`.

In [24]:
df.replace('-', np.nan, inplace=True)
print("'-' values replaced with NaN.")

# Display the count of null values again after replacement
display(df.isnull().sum())

'-' values replaced with NaN.


,0
,38621
Severity,0
Occurred On (NT),0
Cleared On (NT),0
Alarm ID,0
MO Name,0
Comment,99989
Name,0
Location Information,8
Additional Information,21528


Based on the null value counts, we will now drop columns that have a very high percentage of missing values (e.g., more than 90%) as they might not be useful for analysis. We can adjust this threshold if needed.

In [25]:
# Calculate the percentage of missing values for each column
missing_percentages = df.isnull().sum() / len(df) * 100

# Identify columns to drop (e.g., columns with more than 90% missing values)
columns_to_drop = missing_percentages[missing_percentages > 90].index.tolist()

if columns_to_drop:
    print(f"Dropping the following columns due to high missing values: {columns_to_drop}")
    df.drop(columns=columns_to_drop, inplace=True)
    print("Columns dropped successfully.")
    print("New DataFrame shape:", df.shape)
    print("Updated missing value counts (only showing remaining columns with nulls):")
    display(df.isnull().sum()[df.isnull().sum() > 0])
else:
    print("No columns found with more than 90% missing values to drop at this threshold.")



Dropping the following columns due to high missing values: ['Comment', 'Common Alarm Identifier', 'Link Name', 'Link Type', 'Maintenance Region', 'Operation Impact Flag', 'RRU Name', 'Request ID', 'Trouble Ticket ID', 'User Label', 'gNodeB ID']
Columns dropped successfully.
New DataFrame shape: (100000, 33)
Updated missing value counts (only showing remaining columns with nulls):


,0
,38621
Location Information,8
Additional Information,21528
BBU Name,39012
Dev Root CSN,37284
Object Type,25666
eNodeB ID,39196
